In [1]:
import pandas as pd

# Si tu archivo es un CSV (cambia 'tu_archivo.csv' por el nombre real de tu archivo):
df = pd.read_csv('../data/Customer_clean.csv')

# (Si fuera Excel, sería: df = pd.read_excel('tu_archivo.xlsx'))

# Ahora sí revisamos la información:
print(df.info())
print(df.head(3))

<class 'pandas.DataFrame'>
RangeIndex: 793 entries, 0 to 792
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   customer_id    793 non-null    str  
 1   customer_name  793 non-null    str  
 2   segment        793 non-null    str  
 3   age            793 non-null    int64
 4   city           793 non-null    str  
 5   state          793 non-null    str  
 6   postal_code    793 non-null    int64
 7   region         793 non-null    str  
dtypes: int64(2), str(6)
memory usage: 49.7 KB
None
  customer_id        customer_name    segment  age          city  \
0    LH-17020          Lisa Hazard   Consumer   60      Columbus   
1    CK-12205  Chloris Kastensmidt   Consumer   61  Philadelphia   
2    MG-18205      Mitch Gastineau  Corporate   55  Jacksonville   

          state  postal_code region  
0          Ohio        43229   East  
1  Pennsylvania        19120   East  
2       Florida        32216  South  


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer

df = pd.read_csv('../data/Customer_clean.csv')

if 'postal_code' in df.columns and 'zip' not in df.columns:
    df['zip'] = df['postal_code'].astype(str)
elif 'zip' in df.columns:
    df['zip'] = df['zip'].astype(str)

# Si 'country' no existe en el CSV, se puede crear con el valor por defecto o filtrar solo las que existan
available_features = [col for col in ['country', 'state', 'city', 'zip'] if col in df.columns]

X = df[available_features]
y = df['region']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), available_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
X_train_df = pd.DataFrame(X_train_processed, columns=feature_names)

print(f"Dimensiones de X_train procesado: {X_train_df.shape}")
print(f"Distribución de clases en y (region):\n{y.value_counts()}")
print("\nPrimeras filas de X_train procesado:")
print(X_train_df.head(3))

Dimensiones de X_train procesado: (634, 531)
Distribución de clases en y (region):
region
West       255
East       220
Central    184
South      134
Name: count, dtype: int64

Primeras filas de X_train procesado:
   cat__state_Alabama  cat__state_Arizona  cat__state_Arkansas  \
0                 0.0                 0.0                  0.0   
1                 0.0                 0.0                  0.0   
2                 0.0                 0.0                  0.0   

   cat__state_California  cat__state_Colorado  cat__state_Connecticut  \
0                    0.0                  1.0                     0.0   
1                    1.0                  0.0                     0.0   
2                    1.0                  0.0                     0.0   

   cat__state_Delaware  cat__state_District of Columbia  cat__state_Florida  \
0                  0.0                              0.0                 0.0   
1                  0.0                              0.0               

In [ ]:

X_all_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

df_processed = pd.DataFrame(X_all_processed, columns=feature_names)

df_processed['target_region'] = y.values

df_processed.to_csv('../data/Customer_region_preprocessed.csv', index=False)
print("Archivo guardado exitosamente en data/Customer_region_preprocessed.csv")

Archivo guardado exitosamente en data/Customer_region_preprocessed.csv
